# Football Analytics – lokales Notebook

Nutzt dieselben Module wie das Backend (`backend/app`), aber ohne Server: Daten kommen direkt von Fleaflicker und nflverse.

```bash
pip install -r notebooks/requirements.txt
jupyter lab notebooks/analysis.ipynb
```
Caches landen in `../data` (gleicher Ordner wie beim Docker-Container).

In [1]:
# Einmalig: Abhängigkeiten in den aktiven Kernel installieren (dauert beim ersten Mal 1-2 Minuten)
import sys, subprocess
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Python:", sys.executable)
print("Projekt:", ROOT)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "notebooks" / "requirements.txt")], capture_output=True, text=True)
print("pip:", "ok" if r.returncode == 0 else r.stderr[-2000:])

Python: c:\Users\david\workspace\claude_cowork\football_analytics\notebooks\.venv\Scripts\python.exe
Projekt: c:\Users\david\workspace\claude_cowork\football_analytics
pip: ok


In [2]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("DATA_DIR", str(ROOT / "data"))
sys.path.insert(0, str(ROOT / "backend"))

import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

from app import analysis, nflverse, store
from app.fleaflicker import FleaflickerClient, parse_scoring, parse_slots
from app.models import LeagueConfig

LEAGUE_ID = 354024
MY_TEAM_ID = 1838651      # NotBrady's Team
LOOKAHEAD = 5

client = FleaflickerClient()
print("Module geladen, DATA_DIR =", os.environ["DATA_DIR"])

Module geladen, DATA_DIR = c:\Users\david\workspace\claude_cowork\football_analytics\data


## 1. Liga-Konfiguration (Stammdaten)

Wird aus Fleaflicker gelesen und in `data/leagues.json` gespeichert – dieselbe Datei, die auch das GUI nutzt. Slots kannst du hier direkt anpassen.

In [3]:
print("Lade Liga-Regeln von Fleaflicker …")
rules = client.rules(LEAGUE_ID)
print("ok:", len(rules.get("rosterPositions", [])), "Roster-Slots")
standings = client.standings(LEAGUE_ID)
cfg = store.get_league(LEAGUE_ID) or LeagueConfig(id=LEAGUE_ID)
cfg.name = (standings.get("league") or {}).get("name", cfg.name)
cfg.season = standings.get("season")
cfg.my_team_id = MY_TEAM_ID
cfg.lookahead_weeks = LOOKAHEAD
cfg.scoring = parse_scoring(rules)
if not cfg.slots:
    cfg.slots = parse_slots(rules)
store.save_league(cfg)

pd.DataFrame([s.model_dump() for s in cfg.slots])

Lade Liga-Regeln von Fleaflicker …
ok: 17 Roster-Slots


,label,eligibility,start,enabled
0,QB,[QB],1,True
1,RB,[RB],2,True
2,WR,[WR],2,True
3,TE,[TE],1,True
4,RB/WR/TE,"[RB, WR, TE]",4,True
5,QB/RB/WR/TE,"[QB, RB, WR, TE]",1,True
6,K,[K],1,True
7,P,[P],0,False
8,D/ST,[D/ST],0,False
9,CB,[CB],0,False


In [4]:
# Beispiel: Kicker aus der Analyse nehmen
# for s in cfg.slots:
#     if s.label == "K": s.enabled = False
# store.save_league(cfg)
print("Analysierte Positionen:", cfg.enabled_positions())
teams = [(t["id"], t["name"]) for d in standings["divisions"] for t in d["teams"]]
teams

Analysierte Positionen: ['QB', 'RB', 'WR', 'TE', 'K']


[(1825300, 'Boom Shakalaka Bulls'),
 (1830145, "MacDoorbell's Team"),
 (1838651, "NotBrady's Team"),
 (1838656, "Uelu's Team"),
 (1838657, "ShinyNewPanther's Team"),
 (1839954, "TheDoom's Team")]

## 2. Daten laden

Schedule + Wochenstats von nflverse, Roster/Spieler-Pool von Fleaflicker. Erster Lauf lädt ~10–20 MB, danach Cache.

In [5]:
print("Lade nflverse (Schedule + Wochenstats) und Fleaflicker-Roster … beim ersten Mal ~1 Minute")
ctx = analysis.get_context(LEAGUE_ID, client, force=True)
print(f"Saison {ctx.season}, Woche {ctx.week} · Stats {ctx.season}: {len(ctx.cur)} Zeilen · {ctx.season-1}: {len(ctx.prior)} Zeilen · {len(ctx.owned)} Spieler auf Rostern")

Lade nflverse (Schedule + Wochenstats) und Fleaflicker-Roster … beim ersten Mal ~1 Minute
Saison 2026, Woche 1 · Stats 2026: 0 Zeilen · 2025: 6580 Zeilen · 103 Spieler auf Rostern


## 3. Defense vs. Position

Rang 1 = stärkste Defense, 32 = schwächste (bestes Matchup). Punkte nach deinem Liga-Scoring.

In [6]:
def defense_table(ctx):
    rows = []
    for d in ctx.defense:
        r = {"team": d["team"], "run_rank": d["run"]["rank"], "rush_yds": d["run"]["yds_pg"], "rush_td": d["run"]["tds_pg"],
             "pass_rank": d["pass"]["rank"], "pass_yds": d["pass"]["yds_pg"], "pass_td": d["pass"]["tds_pg"]}
        for pos, v in d["vs"].items():
            r[f"pts_{pos}"] = v["pts_pg"]; r[f"rank_{pos}"] = v["rank"]
        r["games"] = f'{d["games_current"]} (+{d["games_prior"]}×{d["prior_weight"]})'
        rows.append(r)
    return pd.DataFrame(rows).set_index("team")

df_def = defense_table(ctx)
df_def.sort_values("run_rank", ascending=False).style.background_gradient(subset=[c for c in df_def.columns if "rank" in c], cmap="RdYlGn")

,run_rank,rush_yds,rush_td,pass_rank,pass_yds,pass_td,pts_QB,rank_QB,pts_RB,rank_RB,pts_WR,rank_WR,pts_TE,rank_TE,pts_K,rank_K,games
team,,,,,,,,,,,,,,,,,
NYG,32,145.100000,1.240000,16,229.900000,1.470000,23.600000,21,25.880000,29,33.660000,24,11.660000,9,8.060000,24,0 (+17×0.5)
CIN,31,146.600000,1.060000,26,245.600000,1.940000,24.580000,25,28.480000,32,25.350000,2,20.980000,32,7.290000,15,0 (+17×0.5)
BUF,30,135.800000,1.410000,1,170.200000,1.120000,16.970000,3,24.040000,24,26.640000,6,7.570000,1,6.060000,5,0 (+17×0.5)
NYJ,29,139.300000,1.180000,20,226.400000,2.120000,26.490000,31,27.860000,31,30.760000,16,15.370000,26,9.350000,32,0 (+17×0.5)
WAS,28,141.600000,1.060000,30,257.100000,1.940000,26.300000,30,25.340000,28,35.080000,25,15.830000,27,7.820000,20,0 (+17×0.5)
DAL,27,125.400000,1.350000,32,265.900000,2.060000,30.810000,32,25.070000,27,39.370000,32,12.460000,13,8.760000,29,0 (+17×0.5)
MIA,26,132.200000,1.000000,19,230.600000,1.710000,24.410000,24,24.560000,26,29.580000,14,16.340000,29,6.940000,12,0 (+17×0.5)
CHI,25,134.500000,0.880000,23,239.200000,1.880000,24.700000,26,21.160000,14,36.210000,30,13.120000,15,6.760000,8,0 (+17×0.5)
ARI,24,126.900000,1.120000,24,242.800000,1.820000,23.320000,19,26.340000,30,30.220000,15,16.940000,31,9.290000,31,0 (+17×0.5)


In [7]:
print("Schwächste Run-Defenses:", list(df_def.sort_values("run_rank", ascending=False).index[:8]))
print("Schwächste Pass-Defenses:", list(df_def.sort_values("pass_rank", ascending=False).index[:8]))

Schwächste Run-Defenses: ['NYG', 'CIN', 'BUF', 'NYJ', 'WAS', 'DAL', 'MIA', 'CHI']
Schwächste Pass-Defenses: ['DAL', 'PIT', 'WAS', 'IND', 'TB', 'BAL', 'CIN', 'TEN']


## 4. Schwache Defense → wer spielt dagegen?

Für eine Woche und Position: die weichsten Defenses und alle Spieler aus dem Pool, die gegen sie spielen (mit Besitzer).

In [ ]:
def matchups(ctx, week, position, top=8):
    pool = [p for p in ctx.player_pool() if p.position == position]
    key = "run" if position == "RB" else "pass"
    order = sorted(ctx.defense, key=lambda d: -(d["vs"][position]["rank"] if position == "K" else d[key]["rank"]))[:top]
    rows = []
    for d in order:
        for p in pool:
            m = ctx.matchup(p.team, position, week)
            if m.get("opponent") != d["team"]:
                continue
            a = ctx.analyze_player(p, [week])
            rows.append({"defense": d["team"], f"{key}_rank": d[key]["rank"], "pts_to_pos": d["vs"][position]["pts_pg"],
                         "player": p.name, "team": p.team, "owner": a["owner_team_name"] or "FA",
                         "score": a["weeks"][0]["score"], "baseline": a["baseline"], "src": a["baseline_source"],
                         "ff_proj": p.projected, "ff_matchup_rank": p.ff_matchup_rank, "inj": p.injury})
    df = pd.DataFrame(rows)
    return df.sort_values([f"{key}_rank", "score"], ascending=[False, False]) if len(df) else df

matchups(ctx, ctx.week, "RB")

In [ ]:
matchups(ctx, ctx.week, "WR")

## 5. Spieler mit Ausblick (Woche N … N+k)

`owner`: `"free"`, `"mine"`, `"all"` oder eine Team-ID.

In [ ]:
def outlook(ctx, owner="free", position=None, from_week=None, weeks=None):
    wks = list(range(from_week or ctx.week, (from_week or ctx.week) + (weeks or ctx.cfg.lookahead_weeks)))
    wks = [w for w in wks if w <= 18]
    rows = []
    for p in ctx.player_pool():
        if position and p.position != position:
            continue
        a = ctx.analyze_player(p, wks)
        oid = a["owner_team_id"]
        if owner == "free" and oid: continue
        if owner == "mine" and oid != ctx.cfg.my_team_id: continue
        if owner not in ("free", "mine", "all") and oid != owner: continue
        r = {"player": p.name, "pos": p.position, "team": p.team, "owner": a["owner_team_name"] or "FA",
             "baseline": a["baseline"], "src": a["baseline_source"], "ff_proj": p.projected, "ff_rank": p.ff_matchup_rank}
        for w in a["weeks"]:
            r[f"W{w['week']}"] = None if w["bye"] else w["score"]
            r[f"W{w['week']}_opp"] = "BYE" if w["bye"] else ("" if w["home"] else "@") + w["opponent"]
        r.update(avg=a["outlook_avg"], total=a["outlook_total"], bye=p.bye_week, inj=p.injury)
        rows.append(r)
    df = pd.DataFrame(rows)
    return df.sort_values("avg", ascending=False) if len(df) else df

df = outlook(ctx, owner="free", position="RB")
score_cols = [c for c in df.columns if c.startswith("W") and not c.endswith("_opp")]
df.head(30).style.background_gradient(subset=score_cols + ["avg"], cmap="RdYlGn", axis=None)

In [ ]:
# Mein Team
outlook(ctx, owner="mine")

In [ ]:
# Roster eines Gegners (Team-ID aus der Liste oben)
# outlook(ctx, owner=1825300)

## 6. Wochen-Heatmap für eine Auswahl

In [ ]:
import matplotlib.pyplot as plt

sel = outlook(ctx, owner="free", position="WR").head(15)
m = sel.set_index("player")[score_cols].astype(float)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(m.values, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(m.columns)), m.columns); ax.set_yticks(range(len(m.index)), m.index)
for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        v = m.values[i, j]
        ax.text(j, i, "BYE" if pd.isna(v) else f"{v:.0f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="Score"); plt.title("Free-Agent WR – Ausblick"); plt.tight_layout()